# 03. Calcul des isochrones

Calcul des isochrones de temps de parcours piéton et cyclable en routage
réel (OSMnx) depuis chaque arrêt de transport en commun structurant, sur
le périmètre de la Métropole Rouen Normandie.

**Prérequis** : exécuter `01_acquisition_donnees_OSM.ipynb` et `02_arrets_TC.ipynb` au préalable  
**Entrées** :
- `data/reseau_pieton_MRN.graphml` — graphe piéton (topologie NetworkX, EPSG:2154)
- `data/reseau_velo_MRN.graphml` — graphe cyclable (topologie NetworkX, EPSG:2154)
- `data/arrets_2026.gpkg` — arrêts TC structurants (situation 2026)

**Constantes** (référence : document de cadrage) :
- Vitesse à pied : 5 km/h
- Vitesse à vélo : 15 km/h
- Seuil d'accessibilité : 15 minutes
- Pas des tranches : 5 minutes

**Projection** : EPSG:2154 (RGF93 / Lambert-93)  
**Sortie** :
- `data/isochrones_2026.gpkg` — polygones d'isochrones dissous par (mode de déplacement × niveau de desserte × tranche 5/10/15 min), livrable cartographique
- `data/acces_noeuds_2026.gpkg` — table du meilleur niveau de desserte atteignable par nœud x niveau du réseau (avec temps de parcours associé), entrée du rattachement des carreaux en `05`

## 1. Imports et paramètres

In [ ]:
from pathlib import Path

import geopandas as gpd
import networkx as nx
import osmnx as ox

print(f"OSMnx version : {ox.__version__}")

# Encodage ordinal de la portée structurante (bac exclu). Réutilisé en 05/06
# « Car express » n'apparaît qu'à l'horizon SERM 2030.
NIVEAUX = {"Bus": 1, "TEOR": 2, "Métro": 3, "Car express": 4, "Train": 5}

# Vitesses de déplacement (km/h) — référence Cerema / INSEE et cycliste urbain ordinaire
VITESSE_PIED = 5
VITESSE_VELO = 15

# Paramètres d'accessibilité (minutes)
SEUIL_ACCESSIBILITE = 15
PAS_TRANCHE = 5

SEUIL_SNAP = 100  # m, aligné sur l'imprécision du carroyage (200 m donc 0–100 m)
TAMPON_ISO = 50  # m, demi-largeur du corridor desservi autour des voies atteintes, impact esthétique uniquement

CRS_PROJET = "EPSG:2154"

Détection de la racine du projet via le fichier sentinelle `.projectroot`,
puis ancrage de tous les chemins dessus. Cette approche est indépendante du
répertoire de travail : elle fonctionne sous JupyterLab comme sous VSCode,
quel que soit le réglage `notebookFileRoot`, et après un clone comme après un
téléchargement ZIP du dépôt.

In [ ]:
def trouver_racine(marqueur=".projectroot"):
    """Remonte depuis le cwd jusqu'au dossier contenant `marqueur`."""
    base = Path.cwd().resolve()
    for parent in (base, *base.parents):
        if (parent / marqueur).exists():
            return parent
    raise FileNotFoundError(f"Racine introuvable (marqueur {marqueur!r})")


ROOT = trouver_racine()
DATA_DIR = ROOT / "data"
print(f"Racine du projet : {ROOT}")

## 2. Chargement des graphes de réseau

Les graphes piéton et cyclable produits par `01_acquisition_donnees_OSM.ipynb`
sont rechargés depuis leurs fichiers `.graphml`, qui conservent la topologie
nécessaire au calcul des plus courts chemins (étapes suivantes).

In [ ]:
G_pieton = ox.load_graphml(DATA_DIR / "reseau_pieton_MRN.graphml")
G_velo = ox.load_graphml(DATA_DIR / "reseau_velo_MRN.graphml")

Contrôle de cohérence : nombre de nœuds/arêtes et système de projection de
chaque graphe. Les deux graphes doivent être projetés en EPSG:2154.

In [ ]:
for nom, G in {"piéton": G_pieton, "vélo": G_velo}.items():
    print(
        f"Réseau {nom:<7} | "
        f"{G.number_of_nodes():>7,} nœuds | "
        f"{G.number_of_edges():>7,} arêtes | "
        f"CRS : {G.graph.get('crs')}"
    )

## 3. Chargement des arrêts de transport en commun

Couche d'arrêts TC structurants pour la situation 2026, produite par
`02_arrets_TC.ipynb`.

In [ ]:
arrets = gpd.read_file(DATA_DIR / "arrets_2026.gpkg")

print(f"{len(arrets):,} arrêts chargés")
print(f"CRS : {arrets.crs}")
arrets.head()

Contrôle de cohérence : la couche des arrêts doit partager le système de
projection des graphes (EPSG:2154) avant tout rattachement spatial des
arrêts au réseau (étape suivante).

In [ ]:
assert arrets.crs is not None, "La couche des arrêts n'a pas de CRS défini."
assert arrets.crs.to_epsg() == 2154, (
    f"CRS inattendu : {arrets.crs.to_epsg()} (attendu : 2154)"
)
print("Arrêts et graphes alignés en EPSG:2154.")

## 4. Pondération temporelle du réseau

Le routage des étapes suivantes raisonne en temps de parcours, non en
distance. Chaque arête des deux graphes reçoit donc un attribut `temps`
(en minutes), dérivé de sa longueur `length` — exprimée en mètres, les
graphes étant projetés en EPSG:2154 — et de la vitesse du mode (§ 1).

La vitesse est tenue constante par mode. On n'emploie pas
`ox.routing.add_edge_speeds`, dont les vitesses par type de voie sont
calibrées pour le trafic motorisé et sans pertinence pour les modes actifs.
L'hypothèse d'une allure uniforme — 5 km/h à pied, 15 km/h à vélo — reste
cohérente avec la granularité de la demande (carroyage 200 m) ; ses
simplifications (relief, temps d'attente aux carrefours non modélisés)
relèvent des Limites du projet.

Conversion appliquée à chaque arête :
`temps (min) = length (m) / (vitesse (km/h) × 1000 / 60)`,
soit 83,33 m/min à pied et 250 m/min à vélo.

In [ ]:
# Association graphe - vitesse, réutilisable pour l'horizon SERM 2030 (nb 06)
RESEAUX = {
    "piéton": (G_pieton, VITESSE_PIED),
    "vélo":   (G_velo,   VITESSE_VELO),
}

for nom, (G, vitesse_kmh) in RESEAUX.items():
    metres_par_min = vitesse_kmh * 1000 / 60
    nx.set_edge_attributes(
        G,
        {
            (u, v, k): donnees["length"] / metres_par_min
            for u, v, k, donnees in G.edges(keys=True, data=True)
        },
        "temps",
    )
    print(f"{nom:<7} : {vitesse_kmh} km/h → {metres_par_min:6.2f} m/min")

Contrôle de cohérence : chaque arête doit porter un temps fini et positif.
La durée de franchissement de l'arête la plus longue donne un ordre de grandeur. 
Un temps anormalement élevé signalerait une arête aberrante (géométrie erronée ou tronçon non découpé) à investiguer avant le routage.

In [ ]:
for nom, (G, _) in RESEAUX.items():
    temps = [donnees["temps"] for *_, donnees in G.edges(data=True)]
    assert len(temps) == G.number_of_edges(), f"Arêtes sans attribut temps ({nom})"
    assert all(t >= 0 for t in temps), f"Temps négatif détecté ({nom})"
    print(
        f"Réseau {nom:<7} | {len(temps):>7,} arêtes pondérées | "
        f"arête la plus longue : {max(temps):.2f} min"
    )

## 5. Rattachement des arrêts au réseau

Le routage part de nœuds du graphe, chaque arrêt est donc rattaché au nœud le plus proche de chaque réseau — un nœud d'origine piéton et un nœud d'origine cyclable, les deux graphes étant distincts.
Le rattachement s'appuie sur la distance euclidienne (graphes et arrêts partagent une projection métrique EPSG:2154, contrôlée au § 3).

`nearest_nodes(..., return_dist=True)` restitue, outre le nœud, la **distance de rattachement** : l'écart entre l'arrêt et son point d'accroche sur le réseau.
Cette distance n'est pas qu'un sous-produit, elle quantifie le biais du rattachement au nœud (*snap-to-node*).
Le voyageur est supposé démarrer au nœud et non à l'arrêt ; ce tronçon d'approche n'est ni routé ni décompté du budget-temps.
Tant qu'il reste petit devant le rayon d'atteinte (1 250 m à pied, 3 750 m à vélo), le biais est négligeable et du même ordre que l'imprécision déjà admise côté demande (carroyage 200 m, soit 0 à 1,2 min).

L'enjeu se concentre sur les modes structurants (TEOR, métro, train, et car express à l'horizon 2030).
Ce sont eux qui portent l'indicateur, et ce sont eux qui peuvent se situer en zone peu maillée — là où le nœud le plus proche est loin, comme l'a laissé entrevoir l'arête de ~3,3 km repérée au § 4.
La distribution des distances de rattachement, lue séparément pour ces modes, décide donc si le snap-to-node simple est défendable ou s'il faut adopter un snap-to-edge.

In [ ]:
SUFFIXE = {"piéton": "pieton", "vélo": "velo"}

X = arrets.geometry.x.to_numpy()
Y = arrets.geometry.y.to_numpy()

for nom, (G, _) in RESEAUX.items():
    noeuds, dists = ox.distance.nearest_nodes(G, X, Y, return_dist=True)
    arrets[f"noeud_{SUFFIXE[nom]}"] = noeuds
    arrets[f"dist_{SUFFIXE[nom]}"] = dists

print("Arrêts rattachés aux deux réseaux.")
arrets[["mode", "noeud_pieton", "dist_pieton", "noeud_velo", "dist_velo"]].head()

Audit des distances de rattachement, par mode et par réseau.
La lecture décisive porte sur les modes structurants (hors bus, qui forme le maillage dense, et hors bac, exclu de la hiérarchie) : leur p95 et leur maximum indiquent si quelques arrêts déterminants s'accrochent loin du réseau.
Le seuil de tolérance est fixé à 100 m — ordre de grandeur de l'imprécision déjà admise côté demande — et la décision se lit sur le 95ᵉ centile, robuste à un arrêt isolé aberrant.

In [ ]:
structurants = ~arrets["mode"].isin(["Bus", "Bac"])

for nom in RESEAUX:
    col = f"dist_{SUFFIXE[nom]}"
    metres_min = (VITESSE_PIED if nom == "piéton" else VITESSE_VELO) * 1000 / 60

    recap = (
        arrets.groupby("mode")[col]
        .agg(n="size", mediane="median", p95=lambda s: s.quantile(0.95), maxi="max")
        .round(1)
    )
    s = arrets.loc[structurants, col]
    verdict = "OK" if s.quantile(0.95) <= SEUIL_SNAP else "À CORRIGER"

    print(f"\n— Réseau {nom} — distance de rattachement (m) —")
    print(recap)
    print(
        f"  Modes structurants : médiane {s.median():.1f} m | "
        f"p95 {s.quantile(0.95):.1f} m ({s.quantile(0.95) / metres_min:.1f} min) | "
        f"max {s.max():.1f} m ({s.max() / metres_min:.1f} min)"
    )
    print(f"  → snap-to-node : {verdict} (seuil p95 = {SEUIL_SNAP} m)")

In [ ]:
# identification de l'arrêt se trouvant à 459,5 m de toute voie accessible aux piétons (a fortiori d'une voie carrossable) 

col = "dist_pieton"
suspect = arrets.loc[arrets[col].idxmax()]
print(suspect[["nom", "mode", "lignes", "operateur", "id_source"]])
print(f"  dist pieton {suspect['dist_pieton']:.1f} m | dist velo {suspect['dist_velo']:.1f} m")
print(f"  X={suspect.geometry.x:.1f}  Y={suspect.geometry.y:.1f}  (EPSG:2154)")

Dans QGIS, l'arrêt nommé "Le Conihout 1183" est dans la presqu'île du Mesnil-sous-Jumièges, sur une arrêt de graphe piéton et vélo, entourée de deux autres arrêts sur la même arrête (les arrêts La Conihout 1827 et Le Conihout 909).
C'est le biais intrinsèque du snap-to-node dans son cas le plus pur : l'arrêt tombe au milieu d'une longue arête, et nearest_nodes ne peut le rattacher qu'à une extrémité. Si l'arête fait ~900 m sans nœud intermédiaire et que l'arrêt est en son milieu, le nœud le plus proche est mécaniquement à environ 450 m alors que le réseau passe à quelques mètres sous l'arrêt.

## 6. Accessibilité par arrêt

Pour chaque arrêt, on détermine les nœuds du réseau atteignables sous le seuil de 15 min depuis son nœud de rattachement (§ 5), avec leur tranche de 5 min.
L'atteignabilité ne dépend que du couple (graphe, nœud d'origine) : elle est indépendante du niveau de desserte, qui est un attribut de l'arrêt rapporté plus tard (§ 7).
Le calcul est donc mené une fois par **nœud d'origine distinct** — plusieurs quais d'un même arrêt partagent souvent leur point de rattachement —, ce qui évite de recalculer des plus courts chemins identiques.

**Méthode : un unique Dijkstra à seuil par origine.**
`single_source_dijkstra_path_length(G, origine, cutoff=15, weight="temps")` restitue en une passe le temps de parcours à tous les nœuds atteignables en ≤ 15 min.
Les tranches 5 / 10 / 15 s'en déduisent par arrondi supérieur : un nœud atteint à *t* minutes appartient à la tranche `5 × ⌈t / 5⌉`, ce qui produit directement des bandes emboîtées sans repasser sur le graphe.

**Pourquoi pas des ego-graphs successifs.**
`nx.ego_graph(G, origine, radius=r, distance="temps")` exécute en interne un Dijkstra à seuil jusqu'à `r`, puis matérialise le sous-graphe correspondant.
Obtenir les trois bandes imposerait trois appels (r = 5, 10, 15), donc trois Dijkstra par origine, recalculant à chaque fois la zone 0–5 puis 0–10 déjà parcourue — environ trois fois plus de calcul de plus courts chemins. S'y ajoute la construction de trois sous-graphes (copies de nœuds et d'arêtes) dont on n'a pas l'usage : seul le dictionnaire `nœud -> temps` est nécessaire pour buffériser les zones atteintes en § 7.
Les deux approches reposent sur le même Dijkstra et donnent le **même résultat**.
Le choix du seuil unique relève de l'efficacité et de la lisibilité, non de la justesse.

In [ ]:
import math

def atteignabilite(G, noeud_origine, cutoff, pas):
    """Tranche de temps (multiple de `pas`, ≤ `cutoff`) de chaque nœud
    atteignable depuis `noeud_origine`, via un unique Dijkstra à seuil.
    L'origine (t = 0) est classée en première tranche."""
    temps = nx.single_source_dijkstra_path_length(
        G, noeud_origine, cutoff=cutoff, weight="temps"
    )
    return {n: max(pas, pas * math.ceil(t / pas)) for n, t in temps.items()}


acces = {}  # acces[mode][nœud_origine] = {nœud_atteint: tranche}
for nom, (G, _) in RESEAUX.items():
    col = f"noeud_{SUFFIXE[nom]}"
    origines = arrets[col].unique()
    acces[nom] = {
        orig: atteignabilite(G, orig, SEUIL_ACCESSIBILITE, PAS_TRANCHE)
        for orig in origines
    }
    couples = sum(len(r) for r in acces[nom].values())
    print(
        f"Réseau {nom:<7} | {len(origines):>5,} nœuds d'origine distincts | "
        f"{couples:>10,} nœuds atteints (cumul tous arrêts)"
    )

Contrôle de cohérence sur un arrêt témoin d'un mode structurant : les nœuds atteints doivent se répartir sur les seules tranches du gradient (5 / 10 / 15), sans valeur hors borne. La tranche reflète la *première* atteinte ; le compte par tranche correspond donc à des anneaux successifs, non à un cumul.

In [ ]:
import pandas as pd

temoin = arrets.loc[arrets["mode"] == "Métro"].iloc[0]
bandes = pd.Series(acces["piéton"][temoin["noeud_pieton"]]).value_counts().sort_index()

print(f"Arrêt témoin : {temoin['nom']} ({temoin['mode']})")
print("Nœuds atteignables à pied, par tranche (min) :")
print(bandes.to_string())

assert set(bandes.index) <= {PAS_TRANCHE, 2 * PAS_TRANCHE, SEUIL_ACCESSIBILITE}, \
    "Tranche hors gradient 5 / 10 / 15"
print("\nTranches conformes au gradient 5 / 10 / 15.")

## 7. Construction des isochrones (polygones)

Les nœuds atteignables (§ 6) sont convertis en zones surfaciques.
Chaque arête dont les deux extrémités sont atteintes sous la tranche considérée est **tamponnée** d'une demi-largeur de corridor, puis l'ensemble est fusionné (`union_all`).
On écarte l'enveloppe convexe, qui surestimerait grossièrement l'emprise en péri-urbain : le tampon de réseau épouse les voies réellement parcourues.
Les tranches étant emboîtées (un nœud atteint à ≤ 5 min l'est aussi à ≤ 10 et ≤ 15), les polygones le sont également - utile en cartographie comme au croisement avec la population (§ 05).

Le **niveau de desserte** de l'arrêt est conservé sur chaque polygone, et doublé d'un **encodage ordinal** (`niveau_ordre`, entier) traduisant la portée structurante `bus < TEOR < métro < car express < train`. Le bac, hors hiérarchie, est exclu.
Le niveau « aucun » (carreau couvert par aucune isochrone) n'est pas matérialisé, c'est l'absence.

La sortie est **dissoute par (mode de déplacement × niveau × tranche)** : une seule emprise par triplet, plutôt que des milliers d'isochrones par arrêt qui se recouvriraient. L'identité de l'arrêt est perdue, mais aucun traitement aval ne s'en sert (§ 05 raisonne en niveaux, non en arrêts).
La dissolution est faite au niveau des nœuds - union des nœuds atteints par niveau avant tamponnage, ce qui donne le même résultat qu'une dissolution de polygones, en bien moins d'unions.

In [ ]:
TRANCHES = list(range(PAS_TRANCHE, SEUIL_ACCESSIBILITE + 1, PAS_TRANCHE))  # [5, 10, 15]
HORIZON = arrets["horizon"].iloc[0]  # paramètre la sortie (réutilisable nb 06)

# Arêtes en GeoDataFrame, une fois par réseau (colonnes u, v pour le filtrage)
edges = {
    nom: ox.graph_to_gdfs(G, nodes=False, edges=True).reset_index()
    for nom, (G, _) in RESEAUX.items()
}

arrets_hierarchises = arrets[arrets["mode"].isin(NIVEAUX)]  # exclut le bac

records = []
for nom, (G, _) in RESEAUX.items():
    e = edges[nom]
    col_noeud = f"noeud_{SUFFIXE[nom]}"
    for niveau_lbl, niveau_ord in NIVEAUX.items():
        origines = arrets_hierarchises.loc[arrets_hierarchises["mode"] == niveau_lbl, col_noeud].unique()
        if len(origines) == 0:
            continue  # ex. car express en 2026
        for tranche in TRANCHES:
            noeuds = set()
            for orig in origines:
                noeuds.update(n for n, t in acces[nom][orig].items() if t <= tranche)
            sel = e[e["u"].isin(noeuds) & e["v"].isin(noeuds)]
            if sel.empty:
                continue
            records.append({
                "mode_deplacement": nom,
                "niveau": niveau_lbl,
                "niveau_ordre": niveau_ord,
                "tranche_min": tranche,
                "horizon": HORIZON,
                "geometry": sel.geometry.buffer(TAMPON_ISO).union_all(),
            })

isochrones = gpd.GeoDataFrame(records, crs=CRS_PROJET)
print(f"{len(isochrones)} isochrones dissoutes (horizon {HORIZON})")

24 (isochrones) = 2 (modes doux) x 4 (modes TC) x 3 (tranches)

Contrôle de cohérence : aire de chaque emprise par niveau et par tranche.
Deux attentes, l'emboîtement (l'aire croît avec la tranche pour un niveau donné : 5 ≤ 10 ≤ 15) et la validité géométrique.
La comparaison *entre* niveaux n'est en revanche pas monotone : un niveau de portée supérieure (train) compte moins d'arrêts qu'un niveau inférieur (bus) et couvre donc une emprise plus petite, c'est attendu, et c'est précisément ce que mesure l'indicateur.

In [ ]:
assert isochrones.geometry.is_valid.all(), "Géométrie(s) invalide(s)"

recap = (
    isochrones.assign(aire_km2=isochrones.geometry.area / 1e6)
    .pivot_table(index=["mode_deplacement", "niveau_ordre", "niveau"],
                 columns="tranche_min", values="aire_km2")
    .round(1)
    .sort_index()
)
print("Aire des emprises (km²) par niveau et tranche :")
print(recap.to_string())

# Emboîtement : pour chaque (mode, niveau), l'aire doit croître avec la tranche
croissant = recap.apply(lambda r: r.dropna().is_monotonic_increasing, axis=1)
assert croissant.all(), "Emboîtement rompu (aire non croissante avec la tranche)"
print("\nEmboîtement des tranches vérifié (5 ≤ 10 ≤ 15).")

Le contrôle est OK

**L'emboîtement par tranche** est vérifié, et les ratios entre tranches sont plausibles. À pied, niveau bus : 166 < 276 < 358 km², soit des incréments décroissants (×1,66 puis ×1,30). C'est la signature attendue d'isochrones qui saturent — les premières minutes ouvrent du territoire vierge, les dernières recouvrent des zones déjà atteintes par des arrêts voisins.

**Le ratio vélo/piéton** est cohérent avec le §6, et il varie intelligemment selon le niveau. Sur le bus à 15 min, vélo/piéton = 576/358 ≈ 1,6 — modeste, parce que le maillage bus est dense et déjà presque saturé à pied, le vélo n'ajoute pas tant de territoire neuf. Sur le train à 15 min, le ratio explose : 115/20 ≈ 5,8. Logique et porteur de sens : les gares sont peu nombreuses et isolées, leurs isochrones piétonnes ne se recouvrent pas, donc passer à vélo (rayon ×3) ouvre du territoire pur sans redondance. Plus le mode est rare et dispersé, plus le vélo démultiplie son emprise. C'est précisément le principe de l'indicateur : le vélo comme rabattement vers les points structurants isolés.

**La hiérarchie des aires** décroît avec la portée, comme annoncé au contrôle du §7. bus > TEOR > métro > train à emprise égale de tranche : le mode le plus structurant couvre le moins de surface, parce qu'il a le moins d'arrêts. Ce n'est pas un défaut, c'est l'objet même de l'analyse, et c'est ce qui rend le « meilleur niveau atteignable » discriminant. Autre remarque : à pied, train (20 km²) < métro (26,5) à 15 min, mais à vélo train (115) > métro (66). L'inversion vient de la géographie, les gares TER rayonnent sur un territoire large et péri-urbain, le métro reste resserré sur l'axe urbain. En vélo, la dispersion des gares devient un avantage de couverture.